# Forward Problem (on individual subject)

- Source Space: Distributed dipole model with cortical constraints (fix orientation to surface normals)
- Head Model: BEM model obtained from FreeSurfer's watershed (wrapped in MNE)
- Electrode alignment: Iterative closest point (ICP) algorithm + closest point projection
- Conductivities: Default from MNE [0.33, 0.006, 0.33]
- Volume Conduction Solution: obtained from OpenMEEG (wrapped in MNE)

# Batch Wise Run

Run the code for all subjects of a given dataset.

## 📦 Imports

In [ ]:
import mindmap as mm 
from pathlib import Path
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- Configuration file (`config`)

🎚️ **<font color=pink>Parameters</font>**
- `spacing`: spacing between sources
- `conductivities`: electrical conductivity between head layers
- minimum skull thickness (`min_skull_thick`): minimum skull thickness to apply when correcting BEM model (inner layer outside outer layer)
- `n_jobs`: number of parrallel processes during source space building

In [ ]:
# config
env      = mm.io.read_config("../configurations/spike.yml")
mm_dir   = Path(env["MM_DIR"])
subjects = env["SUBS"]

# overwrite option
overwrite = True

# parameters
spacing         = 7
conductivities  = [0.3, 0.006, 0.3]
min_skull_thick = 2
n_jobs          = 10

# file names
feeg = lambda sub: mm_dir/sub/"eeg"/f"{sub}_coreg_ave.fif"
fbem = lambda sub: mm_dir/sub/"fwd"/f"{sub}_bem.fif" 
ffwd = lambda sub: mm_dir/sub/"fwd"/f"{sub}_fwd.fif" 

## ♾️ Loop

In [ ]:
for sub in subjects:

    # check folder existence
    eeg_fname = feeg(sub)
    bem_fname = fbem(sub)
    fwd_fname = ffwd(sub)

    exists = mm.utils.files_exists([eeg_fname, bem_fname, fwd_fname], mdir=True)
    if any(exists) and not overwrite:
        print(f"\033[90mSome of the files to generate already exists for subject {sub}\033[0m")
        continue

    # compute forward solution
    try:
        print(f"\033[94mSolving forward problem for subject {sub} ...\033[0m")
        eeg, bem, fwd = mm.forward_problem(sub, spacing, conductivities, min_skull_thick, icp=False, n_jobs=n_jobs, verbose=1)

    except Exception as e:
        print(f"\033[91mFailed to process subject '{sub}' due to: {e}\033[0m")
        continue

    # save
    print(f"Saving results for subject {sub}")
    eeg.save(eeg_fname, overwrite=True, verbose="warning")
    mne.write_bem_solution(bem_fname, bem, overwrite=True)
    mne.write_forward_solution(fwd_fname, fwd, overwrite=True)

    print(f"\033[92mSuccessfully computed forward solution for subject {sub}\033[0m")

# Step Wise Run

Run the code step by step for a single subject.

## 📦 Imports

In [1]:
from pathlib import Path

import mindmap as mm
from mindmap.utils.mri import correct_skull 

import mne
import nibabel as nib

import numpy as np

import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FreeSurfer reconstruction folder: MNE automatically finds data in folder
- T1 weighted images (`T1`): for visualization
- EEG (`eeg_fwd`): for electrode alignment
- Cortical mask (`ctx_mask`): to delete dipoles outside of cortical area
- `cras`: to align nii images with MNE objects

🎚️ **<font color=pink>Parameters</font>**
- `constraint`: source space constraints (cortical constraints)
- `spacing`: spacing between sources
- `conductivities`: electrical conductivity between head layers
- minimum skull thickness (`min_skull_thick`): minimum skull thickness to apply when correcting BEM model (inner layer outside outer layer)
- `n_jobs`: number of parrallel processes during source space building

In [2]:
env = mm.io.read_config("../configurations/simf.yml")
sub = env["SUBS"][0]

# directories
fs_dir = Path(env["FS_DIR"])
mm_dir = Path(env["MM_DIR"])

# data
data = mm.io.read_dataset(sub, "T1", "eeg_fwd", "ctx_mask", "cras")

# parameters
constraint      = {"type":"surf", "mask":data["ctx_mask"]}
spacing         = 7
conductivities  = [0.15, 0.003, 0.15]
min_skull_thick = 2
n_jobs          = 10

DATA SUMMARY
Dataset: simf
Subject: vepcon_sub-10
Retrieved objects:
- eeg_fwd
- T1
- cras
- ctx_mask
Missing objects:


## Source Space

📤 **<font color=pink>Outputs</font>**
- `sources`: MNE object containing dipole locations and dipole distances to each other.

⚙️ **<font color=green>Define sources along the brain surface</font>**

Sources are distributed uniformly along the brain surface (more specifically, the white surface — the boundary between gray and white matter). The number of sources is adjustable based on the `spacing`. We then eliminate any source that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

In [3]:
# surface-based
if constraint["type"] == "surf": # (1 min)
    sources = mne.setup_source_space(
        subject=sub,
        spacing=spacing,
        surface="white",
        add_dist=True,      # geodesic distances between sources
        subjects_dir=fs_dir,
        n_jobs=n_jobs,
        verbose="warning",
    )

# volumetric
elif constraint["type"] == "vol":
    sources = mne.setup_volume_source_space(
        subject=sub,
        pos=spacing,
        subjects_dir=fs_dir,
        verbose="warning"
    )
    
# gray matter constraint 
if constraint["mask"] is not None:
    sources = mm.utils.mask_sources(sources, constraint["mask"], data["cras"])

print("\033[92mSUMMARY\033[0m")
print(f"Gray matter constraint: {constraint['mask'] is not None}\n")
for src in sources:
    print(f"Source space id: {src["id"]}")
    print(f"Type: {src["type"]}")
    print(f"Number of sources: {src["nuse"]}\n")

SUMMARY
Gray matter constraint: True

Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 2529

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 2432



### <font color=yellow>🌟</font> Visualize source space

In [4]:
mm.viz.plot_sources(sources, mask=data["ctx_mask"], cras=data["cras"], mask_opacity=0.2).show()

Widget(value='<iframe src="http://localhost:46127/index.html?ui=P_0x71347a892ae0_0&reconnect=auto" class="pyvi…

## Head Model

📤 **<font color=pink>Outputs</font>** (files in freesurfer reconstruction bem folder)
- `outer_skin.surf`: outer skin surface (scalp)
- `outer_skull_raw.surf`: outer skull surface
- `inner_skull.surf`: inner skull surface
- `brain.surf`: copy of brain surface
- `inner_skull_raw.surf`: if inner skull going out of outer skull (watershed bug), push outside vertices back inside. This file is the original (bugged) surface
- `inner_skull_changes.vtk`: shows changes between raw inner skull and corrected inner skull

⚙️ **<font color=green>Create surfaces (a.k.a boundaries) from T1-weighted MRI using FreeSurfer's implemented Watershed</font>**

In [5]:
mne.bem.make_watershed_bem(
    subject = sub,
    subjects_dir = fs_dir,
    volume = "T1",
    overwrite = True,
    verbose = "warning",
)

### 🌟 Visualize surfaces

In [6]:
names    = ["outer_skin.surf", "outer_skull.surf", "inner_skull.surf"]
images   = []
for n in names:
    rr, tri = nib.freesurfer.read_geometry(fs_dir/sub/"bem"/n)
    rr = rr + data["cras"]
    rr = nib.affines.apply_affine(np.linalg.inv(data["T1"].affine), rr).astype(int)

    vol = np.zeros(data["T1"].shape)
    vol[rr[:,0], rr[:,1], rr[:,2]] = 255

    img = pv.ImageData(dimensions=data["T1"].shape, spacing=(1,1,1), origin=(0,0,0))
    img.point_data["scalars"] = vol.flatten(order="F")

    images.append(img)


volume = pv.ImageData(dimensions=data["T1"].shape, spacing=(1,1,1), origin=(0,0,0))
volume.point_data["scalars"] = data["T1"].get_fdata().flatten(order="F")

plotter = pv.Plotter()
plotter.add_mesh(volume.contour(3), color="white", opacity=0.05)

def create_slice(normal, origin):
    slc_mri    = volume.slice(normal=normal, origin=origin)
    slc_skin   = images[0].slice(normal=normal, origin=origin)
    slc_oskull = images[1].slice(normal=normal, origin=origin)
    slc_iskull = images[2].slice(normal=normal, origin=origin)
    plotter.add_mesh(slc_mri, cmap="bone", name="mri", opacity="linear")
    plotter.add_mesh(slc_skin, cmap="Reds", name="skin", opacity="linear")
    plotter.add_mesh(slc_oskull, cmap="Reds", name="oskull", opacity="linear")
    plotter.add_mesh(slc_iskull, cmap="Greens", name="iskull", opacity="linear")


plotter.add_plane_widget(
    create_slice,
)
plotter.show_grid()
plotter.show_axes()
plotter.show()

Widget(value='<iframe src="http://localhost:46127/index.html?ui=P_0x71347a304c50_1&reconnect=auto" class="pyvi…

⚙️ **<font color=green>Correct surfaces</font>**
The watershed algorithm sometimes generates an inner skull that extends beyond the outer skull. We have implemented some functions to correct this issue.

In [7]:
correct_skull(
    sub,
    offset = 2,
    fs_dir = fs_dir,
    save_arrows = True,
)

# Correct skull sends an "ERROR:root:No points/cells to operate on" when there are no 
# correction to be made. The program will continue without stopping.

⚙️ **<font color=green>Assemble Boudary Element Model (BEM)</font>**
In this section, we apply topological checks to verify if the surfaces are compatible with each other in order to create a conductor model later.

In [8]:
head_model = mne.make_bem_model(
        subject=sub,
        subjects_dir=fs_dir,
        conductivity=conductivities,
        #ico=None,
        verbose="warning",
)

## Register Electrodes
📤 **<font color=pink>Outputs</font>**
- coregistered EEG (`eeg`): EEG with electrodes aligned to outer skin (scalp)

#### 🌟 Visualize alignmnent

In [27]:
points = mm.utils.get_ch_positions(data["eeg_fwd"])
elec   = pv.PolyData(points)
head   = mm.utils.surf_to_polydata(head_model[0]["rr"], head_model[0]["tris"])

plotter = pv.Plotter()
plotter.add_mesh(elec, render_points_as_spheres=True, color="red", name="initial electrodes")
plotter.add_mesh(head, smooth_shading=True, color="lightgray", name="head")
plotter.show()

Widget(value='<iframe src="http://localhost:46127/index.html?ui=P_0x7135ad8997c0_4&reconnect=auto" class="pyvi…

⚙️ <font color=green>**Iterative Closest Point (ICP)**</font>: ICP is an algorithm employed to minimize the difference between two cloud  points.

In [29]:
#points = mne.transforms.apply_trans(trans, elec)
points, _, _ = mm.utils.point_to_plane_icp_with_scaling(points, head, max_iterations=1)
#_, _, points = mm.utils.optimize_scaling_translation(new_points, head)

⚙️ <font color=green>**Projection to Closest Neighbor**</font>: To ensure the electrodes are actually on the scalp, we project them to the nearest point on the scalp.

In [31]:
points = mm.utils.project_points_to_surface(points, head)

#### <font color=yellow>🌟</font> Visualize corrected alignment (see changes in previous plot 😉)

In [32]:
elec_new = pv.PolyData(points)
_ = plotter.add_mesh(elec_new, render_points_as_spheres=True, color="green", name="new electrodes")

🔄 **Update new electrode positions**

In [33]:
channels = dict(zip(data["eeg_fwd"].info["ch_names"], points))
montage  = mne.channels.make_dig_montage(ch_pos=channels, coord_frame="head")
eeg      = data["eeg_fwd"].set_montage(montage)

## Solve Conduction Volume Problem

📤 **<font color=pink>Outputs</font>**
- volume conductor model (`bem`): Completed Boundary Element Method (BEM) model (head surfaces + electrode position + conductivities)
- `forward`: Contains solution (lead field field + source space + some other stuff)

Note: To solve the volume conduction problem, MNE will sometimes delete some sources that prevent the solving (not sure why). The source space inside `forward` is updated with the deleted sources.

⚙️ <font color=green> **Solve the conduction problem and create the Leadfield matrix**</font>: Now that the head model is complete, we can compute the lead field matrix, which describes the projection of the sources onto the electrodes. To do this, we solve the volume conduction problem using OpenMEEG (wrapped inside MNE)

In [34]:
bem = mne.make_bem_solution(head_model)
trans = mne.transforms.Transform(fro="head", to="mri")      # identity matrix

forward = mne.make_forward_solution(
    info=eeg.info,
    trans=trans,
    src=sources,
    bem=bem,
    meg=False,
    eeg=True,
    verbose="warning",
)

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (2562) -> outer skin  (2562) ...
        outer skin  (2562) -> outer skull (2562) ...
        outer skin  (2562) -> inner skull (2562) ...
        outer skull (2562) -> outer skin  (2562) ...
        outer skull (2562) -> outer skull (2562) ...
        outer skull (2562) -> inner skull (2562) ...
        inner skull (2562) -> outer skin  (2562) ...
        inner skull (2562) -> outer skull (2562) ...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


#### <font color=yellow>🌟</font> Visualize Forward solution

In [35]:
mm.viz.plot_leadfield(eeg, forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:46127/index.html?ui=P_0x7134797d6180_5&reconnect=auto" class="pyvi…

## 💾 Save Forward Solution and co-registered EEGs

**BEM and forward solution**

In [36]:
folder_fwd = mm_dir/sub/"fwd" 

if folder_fwd.exists() == False:
    folder_fwd.mkdir(parents=True)

print(f"Saving head model and forward solution for subject: {sub}")
mne.write_bem_surfaces(folder_fwd/f"{sub}_sim_hm-bem.fif", head_model, overwrite=True)
mne.write_forward_solution(folder_fwd/f"{sub}_sim-bem-fwd.fif", forward, overwrite=True)

Saving head model and forward solution for subject: vepcon_sub-10
Overwriting existing file.
Overwriting existing file.
    Write a source space...
    [done]
    Write a source space...
    [done]
    2 source spaces written


**Co-registered EEGs**

In [37]:
folder_eeg = mm_dir/sub/"eeg"

if folder_eeg.exists() == False:
        folder_eeg.mkdir(parents=True)

print(f"Saving co-registered eeg for subject: {sub}")
if env["NAME"] == "spike":
        eeg.save(folder_eeg/f"{sub}_coreg_ave.fif", overwrite=True)
elif env["NAME"] in ["vep", "simf"]:
        eeg.save(folder_eeg/f"{sub}_coreg_null-ave.fif", overwrite=True, verbose="warning")
elif env["NAME"] == "vepcon":
        eeg.save(folder_eeg/f"{sub}_coreg_epo.fif", overwrite=True, verbose="warning")
else:
        raise ValueError(f"Unknown dataset: {env["NAME"]}")

Saving co-registered eeg for subject: vepcon_sub-10


# Vizualize forward solutions

## 📦 Imports

In [41]:
import mindmap as mm
import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- Configuration file (`config`)
- `sub`: subject id

In [50]:
env = mm.io.read_config("../configurations/spike.yml")
sub = env["SUBS"][11]

# data
data = mm.io.read_dataset(sub, "eeg_coreg", "fwd", "bem", "T1", "ctx_mask", "cras")

DATA SUMMARY
Dataset: spike
Subject: sub-pat12
Retrieved objects:
- ctx_mask
- fwd
- T1
- bem
- eeg_coreg
- cras
Missing objects:


### <font color=yellow>🌟</font> Visualize source space

In [51]:
mm.viz.plot_sources(data["fwd"]["src"], mask=data["ctx_mask"], cras=data["cras"]).show()

No cras translation applied


Widget(value='<iframe src="http://localhost:35425/index.html?ui=P_0x768540a08e60_128&reconnect=auto" class="py…

### 🌟 Visualize surfaces

#### <font color=yellow>🌟</font> Visualize EEG coregistration

In [52]:
scalp = data["bem"][0]
scalp = mm.utils.surf_to_polydata(scalp["rr"], scalp["tris"])
elec  = mm.utils.get_ch_positions(data["eeg_coreg"])

plotter = pv.Plotter()
plotter.add_mesh(scalp)
plotter.add_mesh(elec, render_points_as_spheres=True, color="black")
plotter.show()

Widget(value='<iframe src="http://localhost:35425/index.html?ui=P_0x7685407d0770_129&reconnect=auto" class="py…

#### <font color=yellow>🌟</font> Visualize Forward solution

In [53]:
mm.viz.plot_leadfield(data["eeg_coreg"], data["fwd"]).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:35425/index.html?ui=P_0x768540afb200_130&reconnect=auto" class="py…